# Experiment 4: Transfer Learning on CIFAR-10

This notebook contains the complete runnable source for the experiment. Run all cells to regenerate the report figures and measured results.

In [ ]:
"""Experiment 4: transfer learning on CIFAR-10 with MobileNetV2."""

import json
import os
import pickle
import tarfile
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model


ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SEED = 42
NUM_CLASSES = 10
HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 5
BATCH_SIZE = 32
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass


def load_cifar10():
    archive = Path.home() / ".keras" / "datasets" / "cifar-10-batches-py-target_archive"
    if archive.exists():
        with tarfile.open(archive, "r:gz") as tar:
            def read_batch(filename):
                member = tar.getmember(f"cifar-10-batches-py/{filename}")
                with tar.extractfile(member) as stream:
                    record = pickle.load(stream, encoding="bytes")
                images = record[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
                labels = np.asarray(record[b"labels"], dtype="uint8")
                return images, labels

            train_batches = [read_batch(f"data_batch_{index}") for index in range(1, 6)]
            x_train = np.concatenate([batch[0] for batch in train_batches])
            y_train = np.concatenate([batch[1] for batch in train_batches])
            x_test, y_test = read_batch("test_batch")
            return (x_train, y_train), (x_test, y_test)
    return tf.keras.datasets.cifar10.load_data()


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(ROOT / filename, dpi=110, bbox_inches="tight")
    plt.close(fig)


def build_transfer_model(base, dense_units=128, optimizer_name="adam", learning_rate=0.001):
    inputs = Input(shape=(32, 32, 3), name="cifar_input")
    features = base(inputs, training=False)
    features = GlobalAveragePooling2D(name="global_average_pooling")(features)
    features = Dense(dense_units, activation="relu", name="transfer_dense")(features)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="cifar_output")(features)
    model = Model(inputs, outputs, name="mobilenetv2_cifar10")
    if optimizer_name == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def metric_summary(model, x_test, y_test, y_test_one_hot):
    probabilities = model.predict(x_test, batch_size=128, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    return {
        "accuracy": float(accuracy_score(y_test, predictions)),
        "precision": float(precision_score(y_test, predictions, average="macro", zero_division=0)),
        "recall": float(recall_score(y_test, predictions, average="macro", zero_division=0)),
        "f1": float(f1_score(y_test, predictions, average="macro", zero_division=0)),
        "loss": float(model.evaluate(x_test, y_test_one_hot, verbose=0)[0]),
        "predictions": predictions,
        "confusion_matrix": confusion_matrix(y_test, predictions).tolist(),
        "classification_report": classification_report(
            y_test, predictions, target_names=CLASS_NAMES, digits=4, zero_division=0
        ),
    }


def plot_samples(x_train_raw, y_train):
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for index, axis in enumerate(axes.flat):
        axis.imshow(x_train_raw[index])
        axis.set_title(CLASS_NAMES[int(y_train[index])], fontsize=9)
        axis.axis("off")
    fig.suptitle("Sample CIFAR-10 images")
    save_figure(fig, "exp4_plot_01_sample_images.png")


def plot_history(history):
    fig, axis = plt.subplots(figsize=(9, 5))
    axis.plot(history["accuracy"], label="training accuracy", color="steelblue")
    axis.plot(history["val_accuracy"], label="validation accuracy", color="darkorange")
    axis.set_title("MobileNetV2 training and validation accuracy")
    axis.set_xlabel("epoch")
    axis.set_ylabel("accuracy")
    axis.legend()
    axis.grid(alpha=0.3)
    save_figure(fig, "exp4_plot_02_training_accuracy.png")

    fig, axis = plt.subplots(figsize=(9, 5))
    axis.plot(history["loss"], label="training loss", color="steelblue")
    axis.plot(history["val_loss"], label="validation loss", color="darkorange")
    axis.set_title("MobileNetV2 training and validation loss")
    axis.set_xlabel("epoch")
    axis.set_ylabel("categorical cross-entropy")
    axis.legend()
    axis.grid(alpha=0.3)
    save_figure(fig, "exp4_plot_03_training_loss.png")


def plot_confusion(matrix):
    fig, axis = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        np.asarray(matrix),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axis,
    )
    axis.set_title("Confusion matrix after MobileNetV2 fine-tuning")
    axis.set_xlabel("predicted label")
    axis.set_ylabel("true label")
    save_figure(fig, "exp4_plot_04_confusion_matrix.png")


def plot_misclassified(x_test_raw, y_test, predictions):
    wrong = np.flatnonzero(predictions != y_test)[:12]
    fig, axes = plt.subplots(3, 4, figsize=(10, 7))
    for axis, index in zip(axes.flat, wrong):
        axis.imshow(x_test_raw[index])
        axis.set_title(f"true: {CLASS_NAMES[y_test[index]]}\npred: {CLASS_NAMES[predictions[index]]}", fontsize=8)
        axis.axis("off")
    fig.suptitle("Examples of misclassified CIFAR-10 images")
    save_figure(fig, "exp4_plot_05_misclassified_images.png")


def plot_comparison(frozen_metrics, fine_metrics):
    names = ["frozen base", "fine-tuned"]
    values = [frozen_metrics["accuracy"], fine_metrics["accuracy"]]
    fig, axis = plt.subplots(figsize=(7, 4.5))
    bars = axis.bar(names, values, color=["slateblue", "seagreen"], edgecolor="black")
    axis.set_ylim(0, 1)
    axis.set_ylabel("test accuracy")
    axis.set_title("Frozen base versus fine-tuned MobileNetV2")
    for bar, value in zip(bars, values):
        axis.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.3f}", ha="center")
    axis.grid(axis="y", alpha=0.3)
    save_figure(fig, "exp4_plot_06_frozen_finetuned.png")


def plot_hyperparameters(rows):
    labels = [row["label"] for row in rows]
    values = [row["accuracy"] for row in rows]
    fig, axis = plt.subplots(figsize=(10, 5))
    bars = axis.barh(labels, values, color="teal", edgecolor="black")
    axis.set_xlim(0, 1)
    axis.set_xlabel("test accuracy on the fixed study subset")
    axis.set_title("Controlled hyperparameter study")
    for bar, value in zip(bars, values):
        axis.text(value + 0.005, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center", fontsize=8)
    save_figure(fig, "exp4_plot_07_hyperparameter_study.png")


def main():
    print("Loading CIFAR-10...")
    (x_train_raw, y_train), (x_test_raw, y_test) = load_cifar10()
    y_train = y_train.reshape(-1)
    y_test = y_test.reshape(-1)
    x_train = x_train_raw.astype("float32") / 127.5 - 1.0
    x_test = x_test_raw.astype("float32") / 127.5 - 1.0
    y_train_one_hot = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
    y_test_one_hot = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)
    x_fit, x_val = x_train[:-5000], x_train[-5000:]
    y_fit, y_val = y_train_one_hot[:-5000], y_train_one_hot[-5000:]

    print("train shape:", x_train.shape, y_train_one_hot.shape)
    print("test shape :", x_test.shape, y_test_one_hot.shape)
    print("input range for MobileNetV2:", float(x_train.min()), "to", float(x_train.max()))
    plot_samples(x_train_raw, y_train)

    print("Loading ImageNet MobileNetV2 weights...")
    base = MobileNetV2(include_top=False, weights="imagenet", input_shape=(32, 32, 3))
    base.trainable = False
    frozen_model = build_transfer_model(base)
    frozen_start = time.perf_counter()
    frozen_history = frozen_model.fit(
        x_fit,
        y_fit,
        validation_data=(x_val, y_val),
        epochs=HEAD_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=2,
    )
    frozen_time = time.perf_counter() - frozen_start
    frozen_metrics = metric_summary(frozen_model, x_test, y_test, y_test_one_hot)
    frozen_metrics["training_time_seconds"] = frozen_time
    frozen_metrics["parameters"] = int(frozen_model.count_params())
    frozen_predictions = frozen_metrics["predictions"]
    print("frozen accuracy:", frozen_metrics["accuracy"])

    for layer in base.layers:
        layer.trainable = False
    for layer in base.layers[-20:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True
    fine_model = build_transfer_model(base)
    fine_model.set_weights(frozen_model.get_weights())
    fine_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    fine_start = time.perf_counter()
    fine_history = fine_model.fit(
        x_fit,
        y_fit,
        validation_data=(x_val, y_val),
        epochs=FINE_TUNE_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=2,
    )
    fine_time = time.perf_counter() - fine_start
    fine_metrics = metric_summary(fine_model, x_test, y_test, y_test_one_hot)
    fine_metrics["training_time_seconds"] = fine_time
    fine_metrics["parameters"] = int(fine_model.count_params())
    fine_predictions = fine_metrics["predictions"]
    print("fine-tuned accuracy:", fine_metrics["accuracy"])

    plot_history(fine_history.history)
    plot_confusion(fine_metrics["confusion_matrix"])
    plot_misclassified(x_test_raw, y_test, fine_predictions)
    plot_comparison(frozen_metrics, fine_metrics)

    print("Building features for the controlled hyperparameter study...")
    feature_model = Model(base.inputs[0], GlobalAveragePooling2D()(base.output))
    study_indices = np.arange(10000)
    features = feature_model.predict(x_train[study_indices], batch_size=128, verbose=1)
    study_labels = y_train_one_hot[study_indices]
    study_rows = []
    settings = [
        ("baseline: Adam, lr=0.001, 128 units", 128, "adam", 0.001, 32, 5),
        ("learning rate: 0.0001", 128, "adam", 0.0001, 32, 5),
        ("optimizer: SGD", 128, "sgd", 0.001, 32, 5),
        ("dense units: 256", 256, "adam", 0.001, 32, 5),
        ("batch size: 64", 128, "adam", 0.001, 64, 5),
        ("epochs: 10", 128, "adam", 0.001, 32, 10),
    ]
    for label, units, optimizer, learning_rate, batch_size, epochs in settings:
        head = tf.keras.Sequential(
            [
                Input(shape=(features.shape[1],)),
                Dense(units, activation="relu"),
                Dense(NUM_CLASSES, activation="softmax"),
            ]
        )
        if optimizer == "sgd":
            opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
        else:
            opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        head.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
        start = time.perf_counter()
        history = head.fit(features, study_labels, validation_split=0.1, epochs=epochs, batch_size=batch_size, verbose=0)
        elapsed = time.perf_counter() - start
        value = float(history.history["val_accuracy"][-1])
        study_rows.append({"label": label, "accuracy": value, "time_seconds": elapsed, "epochs": epochs})
    plot_hyperparameters(study_rows)

    architecture_table = [
        {"model": "LeNet-5", "parameters": "60K", "implementation": "theoretical reference"},
        {"model": "AlexNet", "parameters": "61M", "implementation": "theoretical reference"},
        {"model": "VGG16", "parameters": "138M", "implementation": "theoretical reference"},
        {"model": "GoogleNet", "parameters": "6.8M", "implementation": "theoretical reference"},
        {"model": "ResNet50", "parameters": "25.6M", "implementation": "theoretical reference"},
        {"model": "MobileNetV2", "parameters": int(fine_model.count_params()), "implementation": "measured in this experiment"},
    ]

    def serialise(metrics):
        result = dict(metrics)
        result.pop("predictions", None)
        return result

    results = {
        "seed": SEED,
        "dataset": {"train_images": 50000, "test_images": 10000, "input_shape": [32, 32, 3], "classes": CLASS_NAMES},
        "model": "MobileNetV2 with ImageNet weights, global average pooling, Dense(128, ReLU), Dense(10, Softmax)",
        "head_epochs": HEAD_EPOCHS,
        "fine_tune_epochs": FINE_TUNE_EPOCHS,
        "batch_size": BATCH_SIZE,
        "frozen": serialise(frozen_metrics),
        "fine_tuned": serialise(fine_metrics),
        "hyperparameter_study": study_rows,
        "architecture_table": architecture_table,
        "classification_report": fine_metrics["classification_report"],
    }
    (ROOT / "experiment_4_results.json").write_text(json.dumps(results, indent=2))
    (ROOT / "experiment_4_classification_report.txt").write_text(fine_metrics["classification_report"])
    print("Results written to experiment_4_results.json")


if __name__ == "__main__":
    main()
